# Imports and client init

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import io
import csv
import pandas as pd
from collections import defaultdict
from dotenv import load_dotenv
from datetime import datetime
import requests
import uuid

from linalgo.hub.client import LinalgoClient
from linalgo.annotate.models import Corpus, Document, Annotation, Entity, AnnotatorFactory, Target
from wsd.load_data import load_data
from lineval.utils import Body
from linalgo.annotate import models
from wsd.load_corpus import load_corpus

In [3]:
load_dotenv()
token = os.getenv('LINHUB_TOKEN')
url = "https://linhub.api.linalgo.com/v1"
client = LinalgoClient(token, url)
jack_org_id = "acf7a1aa-ec18-4fa2-a981-a756bc6e6af2"
test_task_id = "635b8e9d-b590-4222-83a0-b46762a9fa58"
test_id = "6667052e-b464-47a9-beca-dd8df8f8c632"
jack_org = client.get_organization(jack_org_id)

# Load and import Semcor docs (old way)

In [ ]:
#loading candidates
X, y = load_data(lang='fr')

k = len(X)
X_test, y_test = X[:k], y[:k]
len(X_test), len(y_test)

In [ ]:
semcor_corpus = Corpus(name='Semcor_fr', description='The Semcor wsd corpus for French', organization=jack_org)
corpus = client.create_corpus(semcor_corpus, jack_org)

In [ ]:
target = Target()

In [ ]:
# canidates to corpus
grouped_X = defaultdict(list)
for i,row in enumerate(X_test):
    row.lemma_meaning = y_test[i]
    grouped_X[(row.lemma, row.pos)].append(row)
items = grouped_X.items()
docs = []
for g, cands in items:
    contexts = "\n".join([anno.context for anno in cands])
    doc = Document(content=contexts,
                   corpus=semcor_corpus
                   )
    doc_annos = []
    for c in cands:
        anno = Annotation(document=doc,
                          entity=c.lemma_meaning,
                          body=Body(text=c.text, context=c.context),
                          task="task",
                          annotator="none",
                          target={},
                          created=datetime.now())
        doc_annos.append(anno)
    doc.annotations = set(doc_annos)
    docs.append(doc)

semcor_corpus.documents = docs
X_docs = semcor_corpus.documents
len(X_docs)

# Load corpus March (3/3/2025 superseeding)

In [4]:
new_corpus = load_corpus("fr", test_task_id , jack_org_id, token)
new_corpus

Corpus::Semcor_fr

In [10]:
client.create_corpus(corpus = new_corpus, organization = jack_org)

Exception: Request returned status 400, b'<!DOCTYPE html>\n<html lang="en">\n<head>\n  <meta http-equiv="content-type" content="text/html; charset=utf-8">\n  <meta name="robots" content="NONE,NOARCHIVE">\n  <title>TooManyFieldsSent\n          at /v1/corpora/</title>\n  <style type="text/css">\n    html * { padding:0; margin:0; }\n    body * { padding:10px 20px; }\n    body * * { padding:0; }\n    body { font:small sans-serif; background-color:#fff; color:#000; }\n    body>div { border-bottom:1px solid #ddd; }\n    h1 { font-weight:normal; }\n    h2 { margin-bottom:.8em; }\n    h3 { margin:1em 0 .5em 0; }\n    h4 { margin:0 0 .5em 0; font-weight: normal; }\n    code, pre { font-size: 100%; white-space: pre-wrap; word-break: break-word; }\n    summary { cursor: pointer; }\n    table { border:1px solid #ccc; border-collapse: collapse; width:100%; background:white; }\n    tbody td, tbody th { vertical-align:top; padding:2px 3px; }\n    thead th {\n      padding:1px 6px 1px 3px; background:

# Creating task

In [5]:
#Generate a UID for the task
new_task_id = str(uuid.uuid4())
new_task_id

'a9de7dc2-e7ce-4342-a8e1-d9f4de12153b'

In [6]:
exsting_task_id = "d3ce7764-eb85-4999-b965-c028f539ee33"
entities = client.get_task(exsting_task_id).entities

/home/jack/code/JDryv/linalgo/linalgo-sdk/linalgo/hub/client.py:219: UserWarning: Some annotations have no associated document.
  warnings.warn('Some annotations have no associated document.')


In [7]:
def create_task(
    name: str,
    organization: str,
    task_id : str = str(uuid.uuid4()),
    description: str = None,
    corpus_id: str = None,
    entities: list[Entity] = [],
    ) -> None:


    serialized_entities = [entity.id for entity in entities]

    post_url = url + f"/tasks/"
    data  = {
        "id": task_id,
        "name": name,
        "slug": name,
        "organization": organization,
        "description": description,
        "entities": serialized_entities,
        "corpora": [corpus_id],
    }
    client.post(url = post_url, data= data)
    pass

In [ ]:
create_task(name = "test_task_1",
            organization = jack_org_id,
            task_id = new_task_id,
            corpus_id = new_corpus.id,
            entities = entities)

Exception: Request returned status 400, b'{"corpora":["This list may not be empty."]}'

In [ ]:
task = client.get_task(new_task_id, verbose=True)

# Adding documents

In [ ]:
client.add_documents(corpus.documents)

# Getting gold annotator 

In [ ]:
annotators = client.get(url = f"{url}/annotators/?page_size=1000")["results"]
len(annotators)

In [ ]:
for a in annotators:
    if "gold" in a["name"]:
        print(a)

In [ ]:
for a in annotators:
    if "gold" == a["name"] and 77 == a["owner"]:
        gold_anotator_dict = a
        print(gold_anotator_dict)

In [ ]:
gold = AnnotatorFactory.from_dict(gold_anotator_dict)
gold

# adding annotations

In [ ]:
annos = []

for doc in X_docs:
    for anno in doc.annotations:
        anno.annotator = gold
        annos.append(anno)
len(annos)

In [ ]:
annos[0].__dict__

In [ ]:
for anno in annos:
    client.create_annotations(annos)